# 03A — ASOS-Only Model (Phase 1A Early Checkpoint)

Train logistic regression and XGBoost using only ASOS-observable features.
No HRRR needed. Tests whether there is ML signal beyond the simple rule and NBM.

**Lessons applied from NBM benchmark:**
- scale_pos_weight = ~149 (measured class ratio, not estimated)
- station added as a categorical feature (huge variance observed across stations)
- Per-station AUC-PR breakdown reported alongside aggregate

In [ ]:
import sys
sys.path.insert(0, '/Users/dengzhenhua/Library/Python/3.9/lib/python/site-packages')

BASE_DIR = '/Users/dengzhenhua/Desktop/Desktop - MacBook Pro/vibe coding/fogchaser'
RANDOM_SEED = 42

In [ ]:
# Step 1: Load training and test data, build features
import pandas as pd
import numpy as np
import os

DC_STATIONS = ['KDCA', 'KIAD', 'KBWI', 'KGAI', 'KFDK', 'KHEF', 'KNYG', 'KCGS']

train = pd.read_parquet(f'{BASE_DIR}/data/raw/asos/national_train_2018_2023.parquet')
test  = pd.read_parquet(f'{BASE_DIR}/data/raw/asos/dc_metro_test_2024_2026.parquet')

for df in [train, test]:
    df['valid'] = pd.to_datetime(df['valid'], utc=True)
    df['t_td_spread']    = df['tmpf'] - df['dwpf']
    df['wind_speed_mph'] = df['sknt'] * 1.15078
    df['hour_utc']       = df['valid'].dt.hour
    df['month']          = df['valid'].dt.month
    df['hour_sin']  = np.sin(2 * np.pi * df['hour_utc'] / 24)
    df['hour_cos']  = np.cos(2 * np.pi * df['hour_utc'] / 24)
    df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
    df['vsby_km']   = df['vsby'] * 1.60934
    df['is_fog']    = (df['vsby_km'] < 1.0).astype(int)

# Fix: shared station encoding — build category map from union of both datasets.
# pd.Categorical(df['station']).codes called separately per dataframe creates
# inconsistent integer codes (station X might be code 5 in train, code 2 in test).
all_stations_combined = sorted(set(train['station']) | set(test['station']))
station_cat = pd.CategoricalDtype(categories=all_stations_combined, ordered=False)
for df in [train, test]:
    df['station_code'] = df['station'].astype(station_cat).cat.codes

# Note: DC metro stations are not in training (geographic holdout), so their
# station_code values are valid integers but unseen during training. XGBoost
# will extrapolate based on other features — this is expected and intentional.

# persist_vsby_km removed: it equals vsby_km, from which is_fog is derived.
# Including it lets the model trivially learn "if vsby < 1km → fog" at the same
# timestamp — inflates AUC-PR without testing actual forecasting signal.
# lead_time_hours removed: constant 12 for all rows, adds no information.

ASOS_FEATURES = [
    't_td_spread', 'wind_speed_mph',
    'hour_sin', 'hour_cos', 'month_sin', 'month_cos',
    'station_code'   # shared encoding across train+test
]

train_clean = train.dropna(subset=ASOS_FEATURES + ['is_fog'])
test_clean  = test.dropna(subset=ASOS_FEATURES + ['is_fog'])

# Rule 3: verify temporal integrity
assert train_clean['valid'].max() < test_clean['valid'].min(), \
    'Temporal leakage! Training max date overlaps test min date.'
print(f'Temporal integrity OK: train ends {train_clean["valid"].max().date()}, test starts {test_clean["valid"].min().date()}')

X_train = train_clean[ASOS_FEATURES]
y_train = train_clean['is_fog']
X_test  = test_clean[ASOS_FEATURES]
y_test  = test_clean['is_fog']

print(f'Training: {len(X_train):,} rows, {y_train.sum():,} fog ({y_train.mean()*100:.2f}%)')
print(f'Test (DC metro): {len(X_test):,} rows, {y_test.sum():,} fog ({y_test.mean()*100:.2f}%)')

In [ ]:
# Rule 2: null rates before filling
print('Null rates in training features:')
null_rates = X_train.isnull().mean()
for col, rate in null_rates.items():
    flag = ' ⚠️ >10% — STOP' if rate > 0.10 else ''
    print(f'  {col}: {rate*100:.2f}%{flag}')
assert null_rates.max() <= 0.10, f'Column with >{10}% nulls: {null_rates[null_rates>0.10].index.tolist()}'
print('✅ Null rates acceptable')

In [ ]:
# Rule 4: feature distributions train vs test
print('Feature distribution check (train vs test):')
print(f'{"Feature":<22} {"Train mean":>12} {"Test mean":>12} {"Train std":>12} {"Test std":>12} {"Flag"}')
for col in ASOS_FEATURES:
    tm, tstm = X_train[col].mean(), X_test[col].mean()
    ts, tss  = X_train[col].std(),  X_test[col].std()
    ratio = abs(tm) / abs(tstm) if abs(tstm) > 1e-6 else 1.0
    flag = '⚠️ >2x diff' if (ratio > 2 or ratio < 0.5) else ''
    print(f'{col:<22} {tm:>12.3f} {tstm:>12.3f} {ts:>12.3f} {tss:>12.3f} {flag}')

In [ ]:
# Step 2: Baseline rule
from sklearn.metrics import average_precision_score, brier_score_loss, precision_score, recall_score

results = []

def evaluate(y_true, y_prob, model_name, results_list, test_df_ref=None):
    y_pred = (np.array(y_prob) >= 0.5).astype(int)
    metrics = {
        'model':     model_name,
        'auc_pr':    average_precision_score(y_true, y_prob),
        'brier':     brier_score_loss(y_true, y_prob),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall':    recall_score(y_true, y_pred, zero_division=0),
    }
    results_list.append(metrics)
    print(f"\n{model_name}")
    for k,v in metrics.items():
        if k != 'model': print(f'  {k}: {v:.4f}')
    return metrics

baseline_prob = (
    (test_clean['t_td_spread'] <= 2.0) & (test_clean['wind_speed_mph'] < 5.0)
).astype(float)
evaluate(y_test, baseline_prob, 'Baseline Rule', results)

In [ ]:
# Step 3: Logistic regression
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_SEED)
lr.fit(X_train_s, y_train)

lr_prob = lr.predict_proba(X_test_s)[:, 1]
evaluate(y_test, lr_prob, 'Logistic Regression (ASOS only)', results)

In [ ]:
# Step 4: XGBoost — scale_pos_weight from actual training sample (lesson: ~149, not 20-50)
import xgboost as xgb
from sklearn.model_selection import train_test_split

class_ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(f'scale_pos_weight from training sample: {class_ratio:.1f}')

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.1, random_state=RANDOM_SEED, stratify=y_train
)

xgb_asos = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    scale_pos_weight=class_ratio,
    eval_metric='aucpr',
    early_stopping_rounds=20,
    random_state=RANDOM_SEED,
    tree_method='hist',
)
xgb_asos.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=50)

xgb_prob = xgb_asos.predict_proba(X_test)[:, 1]
evaluate(y_test, xgb_prob, 'XGBoost (ASOS only)', results)

In [ ]:
# Rule 8: Physical sense check — top 20 high-confidence fog predictions
top20 = test_clean.copy()
top20['xgb_prob'] = xgb_prob
top20 = top20.nlargest(20, 'xgb_prob')[['station','valid','hour_utc','month','t_td_spread','wind_speed_mph','is_fog','xgb_prob']]
print('\nTop 20 highest-confidence fog predictions (XGBoost):')
print(top20.to_string(index=False))
print('\nPhysical sense check:')
print(f'  Pre-dawn (0-7am): {(top20["hour_utc"] <= 7).sum()}/20')
print(f'  Oct-Mar:          {top20["month"].isin([10,11,12,1,2,3]).sum()}/20')
print(f'  Actually fog:     {top20["is_fog"].sum()}/20')

In [ ]:
# Per-station breakdown (lesson: aggregate hides station variance)
print('\nXGBoost AUC-PR by station:')
test_clean2 = test_clean.copy()
test_clean2['xgb_prob'] = xgb_prob
for stn, grp in test_clean2.groupby('station'):
    y_s = grp['is_fog']
    if y_s.sum() == 0: continue
    auc = average_precision_score(y_s, grp['xgb_prob'])
    print(f'  {stn}: AUC-PR={auc:.3f}  fog_events={y_s.sum()}')

In [ ]:
# Step 5: Checkpoint decision — compare vs NBM benchmark
import json

results_df = pd.DataFrame(results)
baseline_auc = results_df[results_df['model'] == 'Baseline Rule']['auc_pr'].values[0]
best_auc     = results_df[results_df['model'] != 'Baseline Rule']['auc_pr'].max()
improvement  = best_auc - baseline_auc
random_auc   = y_test.mean()

# Load NBM benchmark results for comparison
nbm_results  = pd.read_csv(f'{BASE_DIR}/outputs/nbm_benchmark_results.csv')
nbm_auc      = nbm_results[nbm_results['model'] == 'NBM + Elevation Weight']['auc_pr'].values[0]
beats_nbm    = best_auc > nbm_auc

print('\n' + '='*55)
print('PHASE 1A CHECKPOINT')
print('='*55)
print(results_df.to_string(index=False))
print(f'\nAUC-PR improvement over baseline: {improvement:.4f}')
print(f'Best model vs random ({random_auc:.4f}): {best_auc/random_auc:.1f}x')
print(f'NBM + terrain AUC-PR (Phase 0):  {nbm_auc:.4f}')
print(f'Best ML model AUC-PR (Phase 1A): {best_auc:.4f}')
print(f'ML beats NBM: {"✅ YES" if beats_nbm else "❌ NO"}')

if improvement > 0.05 and best_auc >= 3 * random_auc:
    if beats_nbm:
        print('\n✅ PASS — beats baseline AND NBM. Proceed to Phase 1B (HRRR features).')
    else:
        print('\n⚠️  Beats baseline but not NBM. HRRR features may push above NBM.')
else:
    print('\n❌ STOP — not beating baseline. Investigate before HRRR pipeline.')

In [ ]:
# Step 6: Save results and model
import os

results_df.to_csv(f'{BASE_DIR}/outputs/phase1a_checkpoint_results.csv', index=False)
xgb_asos.save_model(f'{BASE_DIR}/models/xgb_asos_only_v1.json')

# Rule 10: Log to experiment log
log_path = f'{BASE_DIR}/outputs/experiment_log.csv'
import datetime
for r in results:
    r['timestamp'] = datetime.datetime.now().isoformat()
    r['feature_set'] = 'asos_only_with_station'
log_df = pd.DataFrame(results)
if os.path.exists(log_path):
    existing = pd.read_csv(log_path)
    log_df = pd.concat([existing, log_df], ignore_index=True)
log_df.to_csv(log_path, index=False)
print('Results and model saved. Experiment log updated.')